# Fixed windows to running notes

**Guiding question:** Can a running note use the color that the three-word window could not see?

You ended [Notebook 1](01-counts-and-fixed-window-language-models.ipynb) with identical windows
and different required answers. Keep the same eight sentences, six training rows, and two held-out
rows. The only new route is a note passed from one word to the next.

| Attempt | Complaint | Evidence |
|---|---|---|
| Rebuild the window | Color is outside | Final-token accuracy and equality |
| Carry a running note | A route is not a guarantee | Accuracy and swapped-clue probabilities |
| Inspect the note | It may forget later | Recorded state trace and animation |

The [sequence-memory lesson](../01-sequence-memory/README.md) owns why memory fades and why
protected memory controls help. A **vanilla recurrent neural network (RNN)** is the simplest
running-note network: the same learned update reads the current token and previous note at each
step. We do not reteach gates, the learned controls around memory.
No shared state or downloads. [Handwritten companion](02-fixed-windows-to-recurrent-language-models-theory.md).

## 1. Keep the old failure in view

**Predict:** Will refitting the same three-word input remove its blind spot?

**Image pending:** `images/03-evo-03-fading-notecard.png`
*What this will show:* A running note has a route beyond the window, but recall is not guaranteed.

> **Copilot image prompt**
> - **Target file:** `images/03-evo-03-fading-notecard.png`
> - **Teaches:** A running note has a route beyond the window, but recall is not guaranteed.
> - **Aspect ratio:** 16:9
> - **Prompt:** Show two parallel handoffs reading red -> rest -> calmly -> now and blue -> rest -> calmly -> now. At every handoff show an orange card labelled carried state; words blue, warm/cool green. In a caution inset fade red ink over more handoffs, labelled possible forgetting, not measured result. Clean hand-drawn whiteboard style.
> - **Must show:** Current word and previous note enter every update; no future word enters; fading inset labelled conceptual.
> - **Must not show:** invented numbers, unreadable text, decorative clutter, logos, copyrighted characters
> - **Alt text:** A recurrent reader passes a note between words. The note can carry the clue beyond a window but can also forget.

In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)
started = time.perf_counter()

In [ ]:
corpus = [f"{clue} {first} {second} now {answer}".split()
          for clue, answer in [("red", "warm"), ("blue", "cool")]
          for first in ["wait", "rest"] for second in ["quietly", "calmly"]]
train_rows = [r for r in corpus if r[1:3] != ["rest", "calmly"]]
test_rows = [r for r in corpus if r[1:3] == ["rest", "calmly"]]
for row in corpus:
    print(" ".join(row), "|", "held out" if row in test_rows else "training")

IDs still mean addresses. Each retrieves a trainable row, or **embedding**, before either network
uses it. The window joins three rows in order. The running reader updates one note after each row.
Neither receives the answer as part of the input it is graded on.

In [ ]:
vocab = ["<start>"] + sorted({word for row in corpus for word in row})
to_id = {word: i for i, word in enumerate(vocab)}
def encode(rows):
    return torch.tensor([[to_id[word] for word in row] for row in rows])
train_ids, test_ids = encode(train_rows), encode(test_rows)
print("ID addresses:", to_id)

In [ ]:
WINDOW = 3
def window_pairs(rows):
    contexts, answers = [], []
    for row in rows:
        for position in range(1, len(row)):
            prefix = ["<start>"] * WINDOW + row[:position]
            contexts.append([to_id[word] for word in prefix[-WINDOW:]])
            answers.append(to_id[row[position]])
    return torch.tensor(contexts), torch.tensor(answers)
class WindowLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.rows = nn.Embedding(len(vocab), 8)
        self.predictor = nn.Sequential(nn.Linear(WINDOW * 8, 24), nn.Tanh(),
                                       nn.Linear(24, len(vocab)))
    def forward(self, context):
        return self.predictor(self.rows(context).flatten(1))

In [ ]:
window_model = WindowLM()
x, y = window_pairs(train_rows)
optimizer = torch.optim.Adam(window_model.parameters(), lr=0.03)
for step in range(300):
    loss = F.cross_entropy(window_model(x), y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
with torch.no_grad():
    window_probs = window_model(test_ids[:, -4:-1]).softmax(-1)
    window_accuracy = (window_probs.argmax(-1) == test_ids[:, -1]).float().mean().item()
print(f"Fixed-window final-token accuracy: {window_accuracy:.1%}")
print("Takeaway: fitting again does not give the window a missing color.")

**Your turn:** Change only `inspect_row`. Does selecting the other clue change the visible window?

In [ ]:
inspect_row = 0  # CHANGE THIS: 1 selects the other clue
print("Whole prefix:", test_rows[inspect_row][:-1])
print("Window input:", test_rows[inspect_row][-4:-1])

**Reflection / complaint:** A larger answer table is not a longer view. Carry the missing clue.

## 2. Read a word, update the note, predict the next word

**Predict:** After `red rest calmly now`, can a note still distinguish the first word from `blue`?

```text
red  -> note -> rest -> note -> calmly -> note -> now -> note -> warm?
blue -> note -> rest -> note -> calmly -> note -> now -> note -> cool?
```

The note is a **hidden state**, a learned row describing what has been read so far.
The model returns a note at every position. An output layer turns each note into next-token scores.
Input `red wait quietly now` is graded against `wait quietly now warm`. The answer is one position
ahead. This shift prevents the current answer from being its own input.
Both models see the same next-token pairs. Their parameter counts differ. Equal training budgets
do not isolate architecture from all capacity and optimization differences.

In [ ]:
class RunningLM(nn.Module):
    def __init__(self, width=24):
        super().__init__()
        self.rows = nn.Embedding(len(vocab), 8)
        self.reader = nn.RNN(8, width, batch_first=True)
        self.head = nn.Linear(width, len(vocab))
    def forward(self, tokens):
        notes, last = self.reader(self.rows(tokens))
        return self.head(notes), notes

torch.manual_seed(17)
running_model = RunningLM()
optimizer = torch.optim.Adam(running_model.parameters(), lr=0.02)
losses = []
for step in range(350):
    scores, _ = running_model(train_ids[:, :-1])
    loss = F.cross_entropy(scores.flatten(0, 1), train_ids[:, 1:].flatten())
    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(running_model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
with torch.no_grad():
    scores, notes = running_model(test_ids[:, :-1])
    recurrent_probs = scores[:, -1].softmax(-1)
    recurrent_accuracy = (recurrent_probs.argmax(-1) == test_ids[:, -1]).float().mean().item()
    clue_effect = (recurrent_probs[0] - recurrent_probs[1]).abs().max().item()
print(f"RNN training loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"Held-out final-token accuracy: window {window_accuracy:.1%}; RNN {recurrent_accuracy:.1%}")
print(f"Largest final probability change after swapping only the clue: {clue_effect:.6f}")
for name, model in [("window", window_model), ("RNN", running_model)]:
    print(name, "parameters:", sum(p.numel() for p in model.parameters()))
if recurrent_accuracy > window_accuracy:
    print("Takeaway: this run used the clue beyond the fixed window on the tiny held-out pair.")
else:
    print("Takeaway: the note has a route, but this run did not establish an accuracy gain.")

**Walkthrough:** A row lookup supplies each token vector. The RNN reuses its learned update across
four positions. The output layer predicts the following token at each position. One backward pass
teaches the rows, update, and output layer. Clipping caps gradient size before the update; it does
not restore forgotten information.

**Your turn:** Change only `clue_word`. Keep the filler words. Inspect the final choice.

In [ ]:
clue_word = "red"  # CHANGE THIS: "blue"
probe = torch.tensor([[to_id[w] for w in [clue_word, "rest", "calmly", "now"]]])
with torch.no_grad():
    final = running_model(probe)[0][0, -1].softmax(-1)
print("warm:", round(final[to_id["warm"]].item(), 4),
      "cool:", round(final[to_id["cool"]].item(), 4))

**Reflection / complaint:** The result concerns a short, structured task. It is not unlimited memory.

## 3. Watch the two notes separate

**Predict:** Must the notes become identical after the same filler words?
The curve measures how different the learned notes are at each read position. A changed state is
not proof of useful recall. Read it beside final-token accuracy. The animation uses these same
recorded distances, not an invented fade. The static curve preserves the evidence on GitHub.

In [ ]:
distances = (notes[0] - notes[1]).norm(dim=-1).numpy()
labels = ["red / blue", "rest", "calmly", "now"]
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(labels, distances, marker="o", color="#ef8a23")
ax.set(ylabel="Distance between learned notes", title="Same later words, different opening clue")
plt.show()
print("Recorded distances:", distances.round(4).tolist())
print("Takeaway: read state differences alongside behavior; visible difference alone is not recall.")

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.set(xlim=(-0.2, 3.2), ylim=(0, max(distances) * 1.15),
       ylabel="Distance between notes", title="Watch the measured note travel")
ax.set_xticks(range(4), labels)
line, = ax.plot([], [], "o-", color="#ef8a23")
def frame(position):
    line.set_data(range(position + 1), distances[:position + 1])
    return (line,)
animation = FuncAnimation(fig, frame, frames=4, interval=600, blit=True)
plt.close(fig)
display(HTML(animation.to_jshtml()))

**Your turn:** Change only `position_to_read`. Which position has the biggest recorded difference?

In [ ]:
position_to_read = 3  # CHANGE THIS: 0, 1, or 2
print(labels[position_to_read], float(distances[position_to_read]))

**Reflection / complaint:** A route across four words is not evidence across hundreds.

## 4. What the note fixes, and what it does not

A **long short-term memory network (LSTM)** adds learned controls around a protected memory route.
The [existing gate experiments](../01-sequence-memory/01A-sequence-memory-theory.ipynb) measure
that route. They do not promise every trained LSTM beats every RNN. A **gated recurrent unit
(GRU)** is another controlled running-note design. Notebook 3 uses it without making gate formulas
a new prerequisite. LSTM improves fading, not the one-card reader/writer interface.

| Teaching mechanism | Production counterpart | What changes | What stays |
|---|---|---|---|
| Four note updates | Recurrent text processing | Longer texts and larger states | Same update repeats in order |
| Gradient cap | Gradient clipping | Threshold and optimizer | Cap is not a memory repair |
| Held-out filler combination | Evaluation split | More examples and seeds | Do not grade only memorized rows |

### Evidence scorecard

In [ ]:
print({"window_final_accuracy": window_accuracy, "RNN_final_accuracy": recurrent_accuracy,
       "swapped_clue_probability_effect": clue_effect,
       "RNN_parameters": sum(p.numel() for p in running_model.parameters()),
       "window_parameters": sum(p.numel() for p in window_model.parameters())})
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")

### Three-tier coverage ledger

- **Built and measured:** same corpus and split, ordered-window baseline, vanilla RNN training,
  shifted targets, parameter counts, clue intervention, state-distance storyboard and animation.
- **Explained or illustrated:** hidden state, clipping, short-task limits, LSTM's memory-route
  purpose, and why state difference is not sufficient behavioral evidence.
- **Named and out of scope:** gate internals and fading teaching signals (owned by
  [sequence memory](../01-sequence-memory/README.md)); GRU internals; long-context benchmarks;
  bidirectional next-token reading (future answers are not legal evidence).

**Failure checks:** avoid future-word leakage; do not call two test sentences a benchmark;
do not equate nonzero state distance with correct use of a clue.

**Closing complaint:** You can continue one stream. What if the input and answer are different
sequences? Visit the existing [02C reader/writer introduction](../02-tokenization-and-embeddings/02C-context-windows-to-static-embeddings.ipynb),
then [Notebook 3](03-recurrent-encoder-decoder-to-attention.ipynb).